In [1]:
# from huggingface_hub import snapshot_download

# snapshot_download(repo_id="yhaha/EmoVoice-DB", repo_type="dataset", local_dir="./EmoVoice-DB")

In [4]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [10]:
!ls EmoVoice-DB/audio

__MACOSX   disgusted	  fearful.zip  neutral	    sad.zip
angry	   disgusted.zip  happy        neutral.zip  surprised
angry.zip  fearful	  happy.zip    sad	    surprised.zip


In [8]:
# def loop(files):
#     files, _ = files
#     for f in tqdm(files):
#         with zipfile.ZipFile(f, 'r') as zip_ref:
#             zip_ref.extractall('EmoVoice-DB/audio')

# files = glob('EmoVoice-DB/audio/*.zip')
# multiprocessing(files, loop, len(files))

In [25]:
d = []

files = glob('EmoVoice-DB/*.jsonl')
files = [f for f in files if 'laion' not in f]
for f in files:
    with open(f) as fopen:
        for l in fopen:
            l = json.loads(l)
            d.append(l)
len(d)

64200

In [34]:
!mkdir EmoVoice-DB_audio

mkdir: cannot create directory ‘EmoVoice-DB_audio’: File exists


In [35]:
def loop(rows):
    rows, _ = rows
    data = []
    for row in tqdm(rows):
        try:
            f = os.path.join('EmoVoice-DB', row['target_wav'])
    
            audio_np, sr = sf.read(f)
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
    
            audio_filename = f.replace('/', '_').replace('.wav', '.mp3')
            audio_filename = os.path.join('EmoVoice-DB_audio', audio_filename)
            
            t = row['target_text'].strip()
            if len(t) < 2:
                continue
                
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': row['key']
            })

            
        except Exception as e:
            print(e)
            pass
            
    return data
            

In [36]:
data = loop((d[:10], 0))

100%|██████████| 10/10 [00:00<00:00, 18.36it/s]


In [46]:
len(set([d_['target_wav'] for d_ in d]))

22100

In [38]:
data = multiprocessing(d, loop, cores = 20)

100%|██████████| 3210/3210 [03:43<00:00, 14.34it/s]


In [51]:
'_'.join(data[0]['speaker'].split('_')[-2:])

'angry_ash'

In [54]:
for i in range(len(data)):
    data[i]['speaker'] = 'gpt4o_' + data[i]['speaker']

In [64]:
from collections import defaultdict

dedup = defaultdict(list)
for d in data:
    dedup[d['audio_filename']].append(d)

actual_dedup = []
for k, v in dedup.items():
    actual_dedup.append(v[0])

len(actual_dedup)

22100

In [65]:
from datasets import Dataset

dataset = Dataset.from_list(actual_dedup)
dataset[0]

{'audio_filename': 'EmoVoice-DB_audio/EmoVoice-DB_audio_angry_gpt4o_388_angry_ash.mp3',
 'text': 'The kettle SCREAMED as it reached boiling point, mirroring my inner tension.',
 'speaker': 'gpt4o_angry_ash'}

In [66]:
len(data)

64200

In [67]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'EmoVoice-DB')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 96.56ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████| 1.22MB / 1.22MB,   ???B/s  
Processing Files (1 / 1): 100%|██████████| 1.22MB / 1.22MB,  0.00B/s  
New Data Upload: 100%|██████████|  890kB /  890kB,  0.00B/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.48 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/dc2295f33d0d00778be283ef7b0cd7493fc84c8c', commit_message='Upload dataset', commit_description='', oid='dc2295f33d0d00778be283ef7b0cd7493fc84c8c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [68]:
audio_files = [d['audio_filename'] for d in data]

with open('EmoVoice-DB-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [71]:
folders = glob('EmoVoice-DB_audio*')
folders = [f for f in folders if '.zip' not in f]
for f in folders:
    print(f)
    os.system(f'zip -rq {f}.zip {f}')

EmoVoice-DB_audio_neucodec
EmoVoice-DB_audio


In [72]:
from huggingface_hub import HfApi
api = HfApi()

for f in glob('EmoVoice-DB_audio*.zip'):
    api.upload_file(
        path_or_fileobj=f,
        path_in_repo=f,
        repo_id="malaysia-ai/Multilingual-TTS",
        repo_type="dataset",
    )

Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):   4%|▍         | 41.9MB / 1.02GB,   ???B/s  
Processing Files (0 / 1):  17%|█▋        |  178MB / 1.02GB,  679MB/s  
Processing Files (0 / 1):  30%|██▉       |  304MB / 1.02GB,  655MB/s  
Processing Files (0 / 1):  45%|████▍     |  459MB / 1.02GB,  694MB/s  
Processing Files (0 / 1):  55%|█████▍    |  558MB / 1.02GB,  645MB/s  
Processing Files (0 / 1):  68%|██████▊   |  695MB / 1.02GB,  653MB/s  
Processing Files (0 / 1):  75%|███████▍  |  764MB / 1.02GB,  602MB/s  
Processing Files (0 / 1):  85%|████████▍ |  863MB / 1.02GB,  586MB/s  
Processing Files (0 / 1):  92%|█████████▏|  940MB / 1.02GB,  561MB/s  
Processing Files (0 / 1): 100%|█████████▉| 1.02GB / 1.02GB,  543MB/s  
Processing Files (0 / 1): 100%|█████████▉| 1.02GB / 1.02GB,  489MB/s  
Processing Files (0 / 1): 100%|█████████▉| 1.02GB / 1.02GB,  444MB/s  
Processing Files (0 / 1): 100%|█████████▉| 1.02GB / 1.02GB,  408MB/s  
Processing